# Машинное обучение 

## Семинар 3 (градиентный спуск)

На этом семинаре мы обсудим градиентный спуск, его модификации и подбор параметров в них.

В ноутбуке используется библиотека plotly, которая позволяет создавать интерактивные диаграммы. Эти диаграммы не отображаются в nbviewer или на github, поэтому для удобного просмотра лучше скачать ноутбук, открыть локально и перезапустить его.

## Градиентный спуск

Напомним, что в градиентном спуске значения параметров на следующем шаге получаются из значений параметров на предыдущем шаге смещением в сторону антиградиента функционала: 

$$w^{(t)} = w^{(t-1)} - \eta_t \nabla Q(w^{(t-1)}),$$
где $\eta_t$ — длина шага (learning rate) градиентного спуска.

### Асимптотическая сложность 

Оптимальный набор весов для линейной регрессии с точки зрения MSE выглядит как $w = (X^TX)^{-1}X^Ty$. В этой формуле присутствует обращение матрицы $X^TX$ — очень трудоёмкая операция при большом количестве признаков. Нетрудно подсчитать, что сложность вычислений $O(d^3 + d^2 \ell)$. При решении задач такая трудоёмкость часто оказывается непозволительной, поэтому параметры ищут итерационными методами, стоимость которых меньше. Один из них — градиентный спуск.

Градиент MSE записывается как

$$\nabla Q(w) = 2X^T(Xw - y).$$
 
Сложность вычислений в данном случае $O(d \ell)$. Стохастический градиентный спуск отличается от обычного заменой градиента на несмещённую оценку по одному или нескольким объектам. В этом случае сложность становится $O(kd)$, где $k$ — количество объектов, по которым оценивается градиент, $k \ll \ell$. Это отчасти объясняет популярность стохастических методов оптимизации.

### Визуализация траекторий GD и SGD
На простом примере разберём основные тонкости, связанные со стохастической оптимизацией.

Сгенерируем матрицу объекты—признаки $X$ и вектор весов $w_{true}$, вектор целевых переменных $y$ вычислим как $Xw_{true}$ и добавим нормальный шум:

In [ ]:
import numpy as np
import warnings
# Отключаем предупреждения, чтобы вывод ноутбука был чище
warnings.filterwarnings('ignore')

In [ ]:
n_features = 2   # Количество признаков (размерность пространства)
n_objects = 300  # Количество наблюдений (точек данных)

# Фиксируем seed для воспроизводимости результатов
np.random.seed(100)

# Генерируем "истинные" веса из нормального распределения
w_true = np.random.normal(size=(n_features, ))

# Генерируем матрицу признаков X (300 строк, 2 столбца) со значениями от -5 до 5
X = np.random.uniform(-5, 5, (n_objects, n_features))

# Масштабируем признаки, чтобы создать "вытянутую" функцию потерь (эллипс).
# Это усложняет задачу для градиентного спуска.
# Признак 0 умножается на 1, признак 1 умножается на 3 (0*2+1 и 1*2+1)
X *= (np.arange(n_features) * 2 + 1)[np.newaxis, :]

# Генерируем целевую переменную Y: X * w_true + нормальный шум
Y = X.dot(w_true) + np.random.normal(0, 1, (n_objects))

# Инициализируем случайную начальную точку весов w_0, откуда начнется спуск
w_0 = np.random.uniform(-1, 1, (n_features))

Зададим параметры градиентного спуска:

In [ ]:
batch_size = 10
num_steps = 50

Обучим на полученных данных линейную регрессию для MSE при помощи полного градиентного спуска — тем самым получим вектор параметров.

In [ ]:
# Шаг градиентного спуска (learning rate).
# Определяет, насколько сильно мы сдвигаем w в направлении антиградиента на каждом шаге.
step_size = 1e-2  

# Берём копию начального вектора весов w_0, чтобы не изменять его напрямую.
w = w_0.copy()

# Заводим список, куда будем сохранять веса на каждой итерации (для анализа/визуализации траектории).
w_list = [w.copy()]

# Основной цикл градиентного спуска по числу шагов num_steps.
for i in range(num_steps):
    # Считаем градиент функции потерь MSE:
    # grad = 2 / N * X^T (Xw - Y)
    # и обновляем w шагом в сторону минуса градиента (антиградиента).
    w -= 2 * step_size * np.dot(X.T, np.dot(X, w) - Y) / Y.shape[0]
    
    # Сохраняем копию текущих весов, чтобы не потерять историю.
    w_list.append(w.copy())

# Превращаем список весов в numpy-массив формы (num_steps+1, n_features)
# — так удобнее дальше рисовать траекторию и работать с ним.
w_list = np.array(w_list)

In [ ]:
import plotly.graph_objects as go  # Импортируем Plotly для интерактивных графиков (контуры, точки, линии).


def compute_limits(w_list):
    """
    По траектории w_list и истинному вектору w_true
    вычисляем границы по осям для красивого отображения графика.
    """
    # Максимальное отклонение первой координаты траектории от истины w_true[0],
    # умножаем на 1.1, чтобы добавить небольшой отступ.
    dx = np.max(np.abs(w_list[:, 0] - w_true[0])) * 1.1
    # Аналогично для второй координаты.
    dy = np.max(np.abs(w_list[:, 1] - w_true[1])) * 1.1
    
    # Возвращаем диапазоны по x и y вокруг истинного решения.
    return (w_true[0] - dx, w_true[0] + dx), (w_true[1] - dy, w_true[1] + dy)


def compute_levels(x_range, y_range, num: int = 100):
    """
    Строим сетку точек в пространстве (w1, w2) и считаем значение функции потерь
    MSE в каждой точке, чтобы нарисовать линии уровня.
    """
    # Линейно разбиваем диапазоны по x и y на num точек.
    x, y = np.linspace(x_range[0], x_range[1], num), np.linspace(y_range[0], y_range[1], num)
    # Делаем двумерную сетку координат A (w1), B (w2).
    A, B = np.meshgrid(x, y)

    # Массив под значения функции потерь тех же размеров, что и сетка.
    levels = np.empty_like(A)

    # Проходим по всем точкам сетки.
    for i in range(A.shape[0]):
        for j in range(A.shape[1]):
            # Временный вектор весов w_tmp = (w1, w2) из сетки.
            w_tmp = np.array([A[i, j], B[i, j]])
            # Значение MSE: mean((X w_tmp - Y)^2).
            levels[i, j] = np.mean(np.power(np.dot(X, w_tmp) - Y, 2))
            
    # Возвращаем координаты по осям и матрицу значений потерь.
    return x, y, levels


def make_contour(x, y, levels, name: str = None):
    """
    Создаёт контурный график (линии уровня) функции потерь по сетке (x, y).
    """
    return go.Contour(
        x=x,
        y=y,
        z=levels,
        contours_coloring='lines',  # Рисуем только линии, без заливки.
        line_smoothing=1,           # Немного сглаживаем линии.
        line_width=2,
        ncontours=100,              # Количество уровней.
        opacity=0.5,
        name=name
    )


def make_arrow(figure, x, y):
    """
    Рисует стрелку на фигуре from (x, y) в направлении (dx, dy).
    Используется для визуализации (анти)градиентов.
    """
    # Распаковываем: x — начальная точка, dx — прирост по x.
    x, dx = x
    # Аналогично по y.
    y, dy = y
    
    figure.add_annotation(
        x=x + dx,   # Конечная точка по x.
        y=y + dy,   # Конечная точка по y.
        ax=x,       # Начальная точка по x.
        ay=y,       # Начальная точка по y.
        xref='x',
        yref='y',
        text='',    # Подпись не нужна, только стрелка.
        showarrow=True,
        axref='x',
        ayref='y',
        arrowhead=2,
        arrowsize=1,
        arrowwidth=2,
    )


def plot_trajectory(w_list, name):
    """
    Рисует:
    - линии уровня функции потерь;
    - траекторию изменения весов w_list;
    - финальную точку алгоритма;
    - истинный оптимум w_true.
    """
    # 1. Считаем границы области отображения по траектории и истинным весам.
    x_range, y_range = compute_limits(w_list)
    
    # 2. Строим сетку и значения функции потерь для контуров.
    x, y, levels = compute_levels(x_range, y_range)
    
    # 3. Строим контурный график уровней функции потерь.
    contour = make_contour(x, y, levels, 'Loss function levels')

    # 4. Ломаная линия траектории весов (все точки, кроме последней).
    w_path = go.Scatter(
        x=w_list[:, 0][:-1],          # Первая координата w на всех шагах, кроме финального.
        y=w_list[:, 1][:-1],          # Вторая координата.
        mode='lines+markers',         # Линии + маркеры.
        name='W',
        marker=dict(size=7, color='red')
    )

    # 5. Финальная точка (последнее значение весов).
    w_final = go.Scatter(
        x=[w_list[:, 0][-1]],
        y=[w_list[:, 1][-1]],
        mode='markers',
        name='W_final',
        marker=dict(size=10, color='black'),
    )
    
    # 6. Истинное оптимальное значение w_true, помечаем звездочкой.
    w_true_point = go.Scatter(
        x=[w_true[0]],
        y=[w_true[1]],
        mode='markers',
        name='W_true',
        marker=dict(size=10, color='black'),
        marker_symbol='star'
    )
    
    # 7. Собираем всё в одну фигуру.
    fig = go.Figure(data=[contour, w_path, w_final, w_true_point])

    # Настраиваем оси по ранее вычисленным диапазонам.
    fig.update_xaxes(type='linear', range=x_range)
    fig.update_yaxes(type='linear', range=y_range)

    # Заголовок графика (например, 'Gradient descent', 'SGD' и т.п.).
    fig.update_layout(title=name)

    # Общие настройки внешнего вида.
    fig.update_layout(
        autosize=True,
        # width=700,  # Можно раскомментировать для фиксированной ширины.
        margin=dict(
            l=50,
            r=50,
            b=50,
            t=100,
            pad=4
        ),
        paper_bgcolor='LightSteelBlue',
    )

    # Легенда в верхней части, горизонтально.
    fig.update_layout(legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="right",
        x=1
    ))

    # Гарантируем, что легенда будет показана для всех трейсов.
    fig.update_traces(showlegend=True)

    # Показываем интерактивный график.
    fig.show()

In [ ]:
# w_list — это список весов на каждой итерации, который вернул алгоритм
plot_trajectory(w_list, 'Gradient descent')

На лекции обсуждалось, что градиент перпендикулярен линиям уровня. Это объясняет такие зигзагообразные траектории градиентного спуска. Для большей наглядности в каждой точке пространства посчитаем градиент функционала и покажем его направление.

In [ ]:
# Вычисляем границы графика на основе траектории w_list
x_range, y_range = compute_limits(w_list)
# Вычисляем уровни (изолинии) функции потерь для отрисовки
x, y, levels = compute_levels(x_range, y_range)

# Создаем контурный график
contour = make_contour(x, y, levels, 'Loss function levels')
fig = go.Figure(data=[contour])

# --- Визуализация градиентов (стрелок) ---

# Создаем редкую сетку (10x10) для отрисовки стрелок, чтобы не загромождать график
x_smol, y_smol, _ = compute_levels(x_range, y_range, num=10)
# Убираем края, чтобы стрелки не прилипали к границам
x_smol, y_smol = x_smol[1:-1], y_smol[1:-1]
A_smol, B_smol = np.meshgrid(x_smol, y_smol)

# Проходим по каждому узлу сетки
for i in range(A_smol.shape[0]):
    for j in range(A_smol.shape[1]):
        # Текущая точка весов (w1, w2)
        w_tmp = np.array([A_smol[i, j], B_smol[i, j]])
        
        # Вычисляем АНТИградиент (направление спуска) в этой точке
        # Формула: -learning_rate * 2 * X.T @ (X @ w - Y) / N
        # 0.003 — это learning rate для визуализации
        antigrad = -0.003 * np.dot(X.T, np.dot(X, w_tmp) - Y) / Y.shape[0]
        
        # Рисуем стрелку из точки (w_tmp) в направлении антиградиента
        make_arrow(fig, (A_smol[i, j], antigrad[0]), (B_smol[i, j], antigrad[1]))

# Настройки оформления графика (оси, заголовок, размеры)
fig.update_xaxes(type='linear', range=x_range)
fig.update_yaxes(type='linear', range=y_range)
fig.update_layout(title='Antigradient')
fig.update_layout(autosize=True, margin=dict(l=50, r=50, b=50, t=100, pad=4), paper_bgcolor='LightSteelBlue')

fig.show()

Визуализируем теперь траектории стохастического градиентного спуска, повторив те же самые действия, оценивая при этом градиент по подвыборке.

In [ ]:
def calc_grad_on_batch(X, Y, w, batch_size):
    """
    Считает стохастический градиент по случайному подмножеству объектов (батчу)
    для квадратичной функции потерь MSE: L(w) = mean((X w - Y)^2).
    
    X: матрица объектов размера (N, d)
    Y: вектор ответов размера (N,)
    w: текущий вектор весов размера (d,)
    batch_size: сколько объектов взять в батч
    """
    # Случайно выбираем индексы batch_size объектов без возвращения
    sample = np.random.choice(X.shape[0], size=batch_size, replace=False)
    
    # Считаем градиент только на выбранном подмножестве:
    # grad ≈ 2 / batch_size * X_batch^T (X_batch w - Y_batch)
    return 2 * np.dot(X[sample].T, np.dot(X[sample], w) - Y[sample]) / batch_size


In [ ]:
# Шаг градиентного спуска (learning rate).
step_size = 1e-2

# Начинаем с копии начального вектора весов w_0,
# чтобы не портить исходное значение.
w = w_0.copy()

# В этот список будем записывать веса на каждом шаге,
# чтобы потом визуализировать траекторию.
w_list = [w.copy()]

# Основной цикл стохастического градиентного спуска.
for i in range(num_steps):
    # Делаем шаг по направлению минус стохастический градиент:
    # calc_grad_on_batch считает градиент только на случайном батче из данных.
    w -= step_size * calc_grad_on_batch(X, Y, w, batch_size)
    
    # Сохраняем текущее значение весов.
    w_list.append(w.copy())

# Превращаем список в numpy-массив
# (shape: (num_steps + 1, n_features)) для дальнейшего анализа/рисования.
w_list = np.array(w_list)

In [ ]:
plot_trajectory(w_list, 'Stochastic gradient descent')

Как видно, метод стохастического градиента «бродит» вокруг оптимума. Это объясняется подбором шага градиентного спуска $\eta_k$. Дело в том, что для сходимости стохастического градиентного спуска для последовательности шагов $\eta_k$ должны выполняться [условия Роббинса-Монро](https://projecteuclid.org/download/pdf_1/euclid.aoms/1177729586):
$$
\sum_{k = 1}^\infty \eta_k = \infty, \qquad \sum_{k = 1}^\infty \eta_k^2 < \infty.
$$
Интуитивно это означает следующее: (1) последовательность должна расходиться, чтобы метод оптимизации мог добраться до любой точки пространства, (2) но при этом расходиться не слишком быстро.

Попробуем посмотреть на траектории SGD, последовательность шагов которой удовлетворяет условиям Роббинса-Монро:

In [ ]:
# Начальное значение шага градиентного спуска (η0).
step_size_0 = 0.01

# Стартуем из случайной начальной точки w_0.
w = w_0.copy()

# Сохраняем начальное значение весов для последующей визуализации траектории.
w_list = [w.copy()]


# Цикл стохастического градиентного спуска с изменяемым шагом.
for i in range(num_steps):
    # На i‑й итерации шаг уменьшается как η_i = η0 / (i + 1).
    # Так выполняются условия Роббинса–Монро для сходимости SGD.
    step_size = step_size_0 / (i + 1)
    
    # Делаем шаг по направлению минус стохастический градиент,
    # оцененный на случайном батче размера batch_size.
    w -= step_size * calc_grad_on_batch(X, Y, w, batch_size)
    
    # Сохраняем текущее значение весов.
    w_list.append(w.copy())

# Превращаем список в numpy-массив (num_steps + 1, n_features)
# — удобно для последующего анализа и отрисовки.
w_list = np.array(w_list)


In [ ]:
plot_trajectory(w_list, 'Stochastic gradient descent with dynamic learning rate')

Теперь градиентный спуск движется более направленно, но не доходит до оптимума. При достаточно большом количестве шагов мы всё-таки смогли бы попасть в окрестность минимума, но это слишком долго. Попробуем более сложную схему изменения длины шага:
$$
    \eta_t
    =
    \lambda
    \left(
        \frac{s_0}{s_0 + t}
    \right)^p.
$$
Попробуем взять $s_0 = 1$и поэкспериментируем с разными $\lambda$ и $p$.

In [ ]:
def sgd_with_lr_schedule(lambda_param, p=0.5, s_init=1.0, batch_size=10):
    """
    Стохастический градиентный спуск с убывающим шагом:
        η_t = λ * (s0 / (s0 + t))^p

    lambda_param: λ — базовый масштаб шага
    p: степень убывания шага (чем больше p, тем быстрее шаг → 0)
    s_init: s0 — «смещение» по времени, задаёт начальный масштаб
    batch_size: размер батча для оценки градиента
    """
    # Стартуем с начального вектора весов w_0.
    w = w_0.copy()
    # Сохраняем начальные веса в список траектории.
    w_list = [w.copy()]

    # Цикл по числу шагов градиентного спуска.
    for i in range(num_steps):
        # Расписание шага: η_i = λ * (s_init / (s_init + i))^p.
        # При росте i дробь уменьшается, шаг постепенно затухает.
        step_size = lambda_param * np.power(s_init / (s_init + i), p)

        # Обновляем веса шагом по направлению минус стохастический градиент.
        w -= step_size * calc_grad_on_batch(X, Y, w, batch_size)

        # Сохраняем текущие веса в траекторию.
        w_list.append(w.copy())

    # Возвращаем траекторию в виде numpy-массива для последующей визуализации.
    return np.array(w_list)


In [ ]:
w_list = sgd_with_lr_schedule(lambda_param=0.01, p=0.8)
plot_trajectory(w_list, 'SGD with learning rate schedule')

In [ ]:
w_list = sgd_with_lr_schedule(lambda_param=0.01, p=0.5)
plot_trajectory(w_list, 'SGD with learning rate schedule')

In [ ]:
w_list = sgd_with_lr_schedule(lambda_param=0.01, p=0.35)
plot_trajectory(w_list, 'SGD with learning rate schedule')

По сути, коэффициенты в формуле для длины шага являются гиперпараметрами, и их нужно подбирать. Желательно использовать для этого валидационную выборку.

Посмотрим, как размер подвыборки, по которой оценивается градиент, влияет на сходимость.

In [ ]:
w_list = sgd_with_lr_schedule(lambda_param=0.01, p=0.35, batch_size=1)
plot_trajectory(w_list, 'SGD with learning rate schedule')

In [ ]:
w_list = sgd_with_lr_schedule(lambda_param=0.01, p=0.35, batch_size=10)
plot_trajectory(w_list, 'SGD with learning rate schedule')

In [ ]:
w_list = sgd_with_lr_schedule(lambda_param=0.01, p=0.35, batch_size=100)
plot_trajectory(w_list, 'SGD with learning rate schedule')

Вывод, в общем-то, очевидный: чем больше размер подвыборки, тем более стабильная траектория градиентного спуска. Интереснее посмотреть, как это влияет на скорость сходимости.

### Сравнение скоростей сходимости

Изучим, насколько быстро достигают оптимума методы полного и стохастического градиентного спуска. Сгенерируем выборку и построим график зависимости функционала от итерации.

In [ ]:
num_steps = 100
batch_size = 10

In [ ]:
# data generation
n_features = 50
n_objects = 10000

w_true = np.random.uniform(-2, 2, n_features)

X = np.random.uniform(-10, 10, (n_objects, n_features))
Y = X.dot(w_true) + np.random.normal(0, 5, n_objects)

In [ ]:
from scipy.linalg import norm  # Импортируем функцию нормы вектора (евклидова норма).
    

step_size_sgd = 1e-2  # Базовый шаг для SGD (η_sgd).
step_size_gd = 1e-2   # Постоянный шаг для полного GD (η_gd).

# Инициализируем стартовую точку случайно и делаем одинаковой для обоих методов.
w_sgd = np.random.uniform(-1, 1, n_features)
w_gd = w_sgd.copy()

# Сразу считаем начальные значения MSE для SGD и GD (в одной и той же точке).
residuals_sgd = [np.mean(np.power(np.dot(X, w_sgd) - Y, 2))]
residuals_gd = [np.mean(np.power(np.dot(X, w_gd) - Y, 2))]

# Сюда будем записывать нормы градиентов (оценка для SGD, полный градиент для GD).
norm_sgd = []
norm_gd = []


for i in range(num_steps):
    # Затухающий шаг для SGD: η_i = η0 / (i+1)^0.4.
    # При росте i шаг постепенно уменьшается.
    step_size = step_size_sgd / ((i + 1) ** 0.4)
    
    # Сэмпл индексов объектов для оценки нормы стохастического градиента.
    sample = np.random.randint(n_objects, size=batch_size)
    
    # 1) Обновление SGD:
    # делаем шаг по минус стохастическому градиенту (по батчу).
    w_sgd -= step_size * calc_grad_on_batch(X, Y, w_sgd, batch_size)
    
    # Сохраняем текущее значение функции потерь MSE для SGD.
    residuals_sgd.append(np.mean(np.power(np.dot(X, w_sgd) - Y, 2)))
    
    # Считаем норму "батчевого" градиента в текущей точке w_sgd
    # на выборке sample — это оценка нормы стохастического градиента.
    norm_sgd.append(
        norm(np.dot(X[sample].T, np.dot(X[sample], w_sgd) - Y[sample]))
    )
    
    # 2) Обновление полного GD:
    # один шаг по полному градиенту (используются все объекты).
    w_gd -= 2 * step_size_gd * np.dot(X.T, np.dot(X, w_gd) - Y) / Y.shape[0]
    
    # Сохраняем текущее значение функции потерь MSE для полного GD.
    residuals_gd.append(np.mean(np.power(np.dot(X, w_gd) - Y, 2)))
    
    # Считаем норму полного градиента в точке w_gd (по всей выборке).
    norm_gd.append(
        norm(np.dot(X.T, np.dot(X, w_gd) - Y))
    )

In [ ]:
full_gd = go.Scatter(x=np.arange(num_steps+1), y=residuals_gd, name='Full GD')
sgd = go.Scatter(x=np.arange(num_steps+1), y=residuals_sgd, name='SGD')

fig = go.Figure(data=[full_gd, sgd])

fig.update_xaxes(type='linear', range=[-1, num_steps + 1])
fig.update_yaxes(type='linear')

fig.update_layout(title = 'Residuals comparison', xaxis=dict(title="Iteration"))

fig.update_layout(
    autosize=True,
    width=700,
    margin=dict(
        l=50,
        r=50,
        b=50,
        t=100,
        pad=4
    ),
    paper_bgcolor='LightSteelBlue',
)

fig.show()

In [ ]:
full_gd = go.Scatter(x=np.arange(num_steps+1), y=norm_gd, name='Full GD')
sgd = go.Scatter(x=np.arange(num_steps+1), y=norm_sgd, name='SGD')

fig = go.Figure(data=[full_gd, sgd])

fig.update_xaxes(type='linear', range=[-1, num_steps + 1])
fig.update_yaxes(type='linear')

fig.update_layout(title = 'Gradient norm comparison', xaxis=dict(title="Iteration"))

fig.update_layout(
    autosize=True,
    width=700,
    margin=dict(
        l=50,
        r=50,
        b=50,
        t=100,
        pad=4
    ),
    paper_bgcolor='LightSteelBlue',
)

fig.show()

Как видно, GD буквально за несколько итераций оказывается вблизи оптимума, в то время как поведение SGD может быть весьма нестабильным. Как правило, для более сложных моделей наблюдаются ещё большие флуктуации в зависимости качества функционала от итерации при использовании стохастических градиентных методов. Путём подбора величины шага можно добиться лучшей скорости сходимости, и существуют методы, адаптивно подбирающие величину шага (AdaGrad, Adam, RMSProp).

Ещё интересно посмотреть, как сильно использование mini-batch GD ускоряет сходимость. Посчитаем, за сколько шагов стохастический градиентный спуск достаточно сильно сближается с истинным решением в зависимости от размера батча.

In [ ]:
# Базовые шаги (learning rate) для SGD и для полного GD.
step_size_sgd = 1e-2
step_size_gd = 1e-2
num_steps = 500  # Максимальное число шагов итераций.

# Одна и та же начальная точка для обоих методов.
w_init = np.random.uniform(-1, 1, n_features)
w_gd = w_init.copy()

# Сначала запускаем полный градиентный спуск, чтобы получить «почти-оптимальное» решение.
for i in range(num_steps):
    w_gd -= 2 * step_size_gd * np.dot(X.T, np.dot(X, w_gd) - Y) / Y.shape[0]

# Ошибка в точке после полного GD — целевой уровень качества,
# с которым будем сравнивать SGD.
best_error = np.mean(np.power(np.dot(X, w_gd) - Y, 2))

# Сюда будем складывать количество шагов до «сходимости» для разных batch_size.
steps_before_conv = []

# Набор размеров батча, по которым проводим эксперимент.
batch_sizes = np.arange(0, 500, 10)

for batch_size in batch_sizes:
    # Для каждого размера батча стартуем SGD из той же начальной точки.
    w_sgd = w_init.copy()
    for i in range(num_steps):
        # Затухающий шаг SGD: η_i = η0 / (i+1)^0.4.
        step_size = step_size_sgd / ((i + 1) ** 0.4)
        # Индексы объектов для стохастического градиента (здесь sample дальше не используется).
        sample = np.random.randint(n_objects, size=batch_size)

        # Шаг SGD по батчу.
        w_sgd -= step_size * calc_grad_on_batch(X, Y, w_sgd, batch_size)
        # Текущая MSE‑ошибка.
        err = np.mean(np.power(np.dot(X, w_sgd) - Y, 2))

        # Критерий «достаточного приближения»:
        # как только ошибка отличается от best_error меньше, чем на 1, считаем, что сошлись.
        if np.abs(err - best_error) < 1:
            break

    # Сохраняем номер итерации, на которой алгоритм остановился (число шагов до сходимости).
    steps_before_conv.append(i)


In [ ]:
conv_speed = go.Scatter(x=batch_sizes, y=steps_before_conv, name='Number of steps to convergence')

fig = go.Figure(data=conv_speed)

fig.update_layout(title='Convergence speed',
                 xaxis=dict(title="batch size"),
                yaxis=dict(title="steps before convergence")
)

fig.update_layout(
    autosize=True,
    width=700,
    margin=dict(
        l=50,
        r=50,
        b=50,
        t=100,
        pad=4
    ),
    paper_bgcolor='LightSteelBlue',
)

fig.show()

Видно, что конкретно на этом наборе данных увеличение размера батча примерно до 100 позволяет добиться существенного ускорения сходимости. В то же время увеличение батча в 100 раз приводит и к пропорциональному замедлению каждого шага градиентного спуска. Поэтому, как правило, имеет смысл оценивать градиент по небольшой подвыборке.

### Качество оценки градиента

Интересно посмотреть, как соотносятся стохастический и полный градиенты. Для этого на каждой итерации градиентного спуска будет вычислять полный градиент и смотреть, какой косинус угла в среднем оказывается между ним и стохастической оценкой. Чтобы было с чем сравнить, посчитаем также средний косинус получается между полным градиентом и случайным вектором.

In [ ]:
from scipy.spatial import distance  # Для вычисления косинусного расстояния между векторами.


def plot_angles(step_size_sgd=1e-2, step_size_gd=1e-2, batch_size=1, num_steps=200):
    # Стартовая точка общая для всех экспериментов в этой функции.
    w_init = np.random.uniform(-1, 1, n_features)
    w_sgd = w_init.copy()

    # Сюда будем накапливать средний косинус между stoch_grad и full_grad
    # и между случайным вектором и full_grad на каждой итерации.
    mean_cosine_between_grads = []
    mean_cosine_between_rand = []

    for i in range(num_steps):
        # Полный градиент MSE по всей выборке в текущей точке w_sgd.
        full_grad = 2 * np.dot(X.T, np.dot(X, w_sgd) - Y) / Y.shape[0]

        cosine_between_grads = []
        cosine_between_rand = []
        # Много раз (1000) считаем стохастический градиент и случайный вектор,
        # чтобы получить хорошую оценку среднего косинуса.
        for i in range(1000):
            # Стохастический градиент по батчу размера batch_size.
            stoch_grad = calc_grad_on_batch(X, Y, w_sgd, batch_size)
            # Случайный вектор той же размерности, что и градиент.
            random_vector = np.random.normal(0, 1, full_grad.shape)

            # distance.cosine(a, b) = 1 - cos(a, b),
            # поэтому cos = 1 - distance.cosine(a, b).
            cosine_between_grads.append(-distance.cosine(stoch_grad, full_grad) + 1)
            cosine_between_rand.append(-distance.cosine(random_vector, full_grad) + 1)

        # Среднее по 1000 прогонов — оценка «типичного» косинуса на этой итерации.
        mean_cosine_between_grads.append(np.mean(cosine_between_grads))
        mean_cosine_between_rand.append(np.mean(cosine_between_rand))

        # Обновляем веса шагом SGD с затухающим шагом η_i = η0 / (i+1)^0.4.
        step_size = step_size_sgd / ((i + 1) ** 0.4)
        sample = np.random.randint(n_objects, size=batch_size)
        w_sgd -= step_size * calc_grad_on_batch(X, Y, w_sgd, batch_size)
        
    # Линия: средний косинус между stoch_grad и full_grad.
    cos_grad = go.Scatter(
        x=np.arange(num_steps + 1),
        y=mean_cosine_between_grads,
        name='Cosine between stochastic and full gradients'
    )
    # Линия: средний косинус между случайным вектором и full_grad.
    cos_rand = go.Scatter(
        x=np.arange(num_steps + 1),
        y=mean_cosine_between_rand,
        name='Cosine between random vector and full gradient'
    )

    # Фигура с двумя кривыми.
    fig = go.Figure(data=[cos_grad, cos_rand])

    fig.update_layout(
        title='Stochastic gradient estimate',
        xaxis=dict(title="Iteration")
    )

    fig.update_layout(
        autosize=True,
        width=700,
        margin=dict(
            l=50,
            r=50,
            b=50,
            t=100,
            pad=4
        ),
        paper_bgcolor='LightSteelBlue',
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.02,
            xanchor="right",
            x=1
        )
    )

    fig.show()


In [ ]:
plot_angles(batch_size=1)

Видно, что в целом оценка по одному объекту стабильно лучше, чем движение в случайном направлении. Конечно, оценка градиента по одному объекту даёт оценки, которые достаточно сильно отклоняются от оптимального направления. Но, тем не менее, этого достаточно, чтобы попасть в окрестность оптимума.

А что если оценивать градиент по нескольким объектам?

In [ ]:
plot_angles(batch_size=10)

In [ ]:
plot_angles(batch_size=50)

Видно, что если оценивать градиент по подвыборке, то (а) качество оценки в целом выше и (б) чем мы ближе к оптимуму, тем хуже эта оценка. Можно сделать вывод, что в начале градиентного спуска надо идти в одну и ту же сторону и для оптимизации средней ошибки на всей выборки, и для оптимизации ошибки на отдельных объектах. А вот около оптимума эти направления уже начинают различаться: для уменьшения ошибки на отдельных объектах надо отходить от весов, оптимальных с точки зрения ошибки на всей выборке.